In [4]:
#现在数据的格式
'''
{
  "instruction": "{'instruction': '你是专门进行实体抽取的专家。请从input中抽取出符合schema定义的实体，不存在的实体类型返回空列表。请按照JSON字符串的格式回答。', 'schema': ['name', 'organization', 'position', 'scene'], 'input': '09科特布斯vs斯图加特'}",
  "input": "",
  "output": "{\"name\": [], \"organization\": [\"科特布斯\", \"斯图加特\"], \"position\": [], \"scene\": []}"
}
'''
#改成
'''
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"name\", \"organization\", \"position\", \"scene\"]\ntext: 09科特布斯vs斯图加特"
    }
  ],
  "answer": {
    "name": [],
    "organization": ["科特布斯", "斯图加特"],
    "position": [],
    "scene": []
  },
  "schema": ["name", "organization", "position", "scene"],
  "text": "09科特布斯vs斯图加特"
}
'''
import json
import ast
import os


SYSTEM_PROMPT = "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"


def normalize_answer(answer_obj, schema):
    """
    统一输出格式：
    1. 最终一定返回 dict
    2. key 以 schema 为准
    3. 每个 value 都整理成 list
    4. list 去重并排序
    """

    def norm_list(v):
        if isinstance(v, list):
            return sorted(list(dict.fromkeys(v)))
        elif v is None:
            return []
        else:
            return [v]

    # 情况1：本来就是标准 dict
    if isinstance(answer_obj, dict):
        normalized = {}
        for key in schema:
            normalized[key] = norm_list(answer_obj.get(key, []))
        return normalized

    # 情况2：output 被解析成 list
    # test.json 里存在这种格式：
    # [{"entity": "台湾", "entity_type": "address"}, ...]
    elif isinstance(answer_obj, list):
        normalized = {key: [] for key in schema}

        for item in answer_obj:
            if isinstance(item, dict) and "entity_type" in item and "entity" in item:
                entity_type = item["entity_type"]
                entity = item["entity"]
                # 只保留当前 schema 中定义的实体类型
                if entity_type in normalized:
                    normalized[entity_type].append(entity)
            else:
                raise ValueError(
                    f"answer list 元素格式异常: {repr(item)[:200]}"
                )

        for key in schema:
            normalized[key] = norm_list(normalized[key])
        return normalized

    else:
        raise ValueError(
            f"answer 类型异常: {type(answer_obj)}, 内容: {repr(answer_obj)[:200]}"
        )

def build_user_content(instruction_text, schema, text):
    # 把原始 instruction 稍微整理一下，不保留“请按照JSON字符串格式回答”这种重复表述
    user_instruction = "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。"
    return (
        f"{user_instruction}\n\n"
        f"schema: {json.dumps(schema, ensure_ascii=False)}\n"
        f"text: {text}"
    )


def convert_one(record):
    """
    输入原始样本:
    {
      "instruction": "{'instruction': '...', 'schema': [...], 'input': '...'}",
      "input": "",
      "output": "{\"name\": [], ...}"
    }

    输出 GRPO 样本:
    {
      "prompt": [...],
      "answer": {...},
      "schema": [...],
      "text": "..."
    }
    """
    meta = ast.literal_eval(record["instruction"])
    instruction_text = meta["instruction"]
    schema = meta["schema"]
    text = meta["input"]

    answer = json.loads(record["output"])
    answer = normalize_answer(answer, schema)

    user_content = build_user_content(instruction_text, schema, text)

    grpo_item = {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        "answer": answer,
        "schema": schema,
        "text": text
    }
    return grpo_item


def convert_file(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    total = 0
    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            line = line.strip()
            if not line:
                continue

            record = json.loads(line)
            grpo_item = convert_one(record)

            fout.write(json.dumps(grpo_item, ensure_ascii=False) + "\n")
            total += 1

    print(f"转换完成，共 {total} 条")
    print(f"输出文件: {output_path}")

    # 打印一条样例看看
    with open(output_path, "r", encoding="utf-8") as f:
        first = json.loads(f.readline())
        print("\n示例输出:")
        print(json.dumps(first, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    # 改成你的实际路径
    input_path = ["llm_ner_dataset2/dev.json", "llm_ner_dataset2/test.json", "llm_ner_dataset2/train.json"]
    output_path = ["llm_ner_dataset2/output/dev.json", "llm_ner_dataset2/output/test.json", "llm_ner_dataset2/output/train.json"]
    for inp, out in zip(input_path, output_path):
        convert_file(inp, out)

转换完成，共 3496 条
输出文件: llm_ner_dataset2/output/dev.json

示例输出:
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"address\", \"book\", \"company\", \"game\", \"government\", \"movie\"]\ntext: 09科特布斯vs斯图加特"
    }
  ],
  "answer": {
    "address": [],
    "book": [],
    "company": [],
    "game": [],
    "government": [],
    "movie": []
  },
  "schema": [
    "address",
    "book",
    "company",
    "game",
    "government",
    "movie"
  ],
  "text": "09科特布斯vs斯图加特"
}
转换完成，共 2686 条
输出文件: llm_ner_dataset2/output/test.json

示例输出:
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"position\", \"organization\", \"address\", \"company\", \"book\", \"game\"]\ntext: 彭小军认为，